In [1]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.concat(
    [
        pd.DataFrame(iris.data, columns = iris.feature_names),
        pd.DataFrame(iris.target, columns = ['iris_type'])
    ],
    axis = 1
)

df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),iris_type
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range = (0, 1))

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

models = {
    "Logistic Regression": {
        "model": LogisticRegression(random_state = 42),
        "parameters": {
            "max_iter": [50, 100, 200]
        }
    },
    "SVM": {
        "model": SVC(random_state = 42),
        "parameters": {
            "kernel": ['linear', 'poly', 'rbf', 'sigmoid'],
            "C": [0.1, 1, 10, 100],
            "gamma": ['scale', 'auto'],
            "max_iter": [50, 100, 200]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state = 42),
        'parameters': {
            "criterion": ['gini', 'entropy', 'log_loss'],
            "n_estimators": [10, 14, 30, 50, 100],
            "max_depth": [None, 1, 2, 4, 7, 10, 20],
            "min_samples_split": [2, 3, 4],
            "min_samples_leaf": [2, 3, 4],
            "max_features": ['sqrt', 'log2']
        }
    },
    "Naive Bayes": {
        "model": MultinomialNB(),
        "parameters": {

        }
    }
}

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline

grid_scores = []
randomized_scores = []

for model, config in models.items():
    pipe = Pipeline([
        ("scaler", scaler),
        ("model", config["model"])
    ])

    parameters = {
        f"model__{param}": values for param, values in config["parameters"].items()
    }

    grid = GridSearchCV(
        pipe,
        parameters,
        cv = 5
    )

    randomized = RandomizedSearchCV(
        pipe,
        parameters,
        cv = 5,
        n_iter = 15,
        random_state = 42 # Just to keep randomness fixed throughout my learning journey, like in these models or any other programs
    )

    grid.fit(df.drop("iris_type", axis = 1), df["iris_type"])

    randomized.fit(df.drop("iris_type", axis = 1), df["iris_type"])

    grid_scores.append(
        pd.DataFrame({
            'Model': model,
            'Parameters': grid.cv_results_['params'],
            'Mean CV Score': grid.cv_results_['mean_test_score'],
            'Std CV Score': grid.cv_results_['std_test_score']
        })
    )

    randomized_scores.append(
        pd.DataFrame({
            'Model': model,
            'Parameters': randomized.cv_results_['params'],
            'Mean CV Score': randomized.cv_results_['mean_test_score'],
            'Std CV Score': randomized.cv_results_['std_test_score']
        })
    )

    # it's processor's nightmare XD

c:\Users\bido7\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 3 is smaller than n_iter=15. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\bido7\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\bido7\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\bido7\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warning

In [5]:
grid_scores_df = pd.concat(grid_scores, ignore_index = True)
randomized_scores_df = pd.concat(randomized_scores, ignore_index = True)

In [6]:
grid_scores_df

,Model,Parameters,Mean CV Score,Std CV Score
0,Logistic Regression,{'model__max_iter': 50},0.926667,0.057349
1,Logistic Regression,{'model__max_iter': 100},0.926667,0.057349
2,Logistic Regression,{'model__max_iter': 200},0.926667,0.057349
3,SVM,"{'model__C': 0.1, 'model__gamma': 'scale', 'mo...",0.920000,0.045216
4,SVM,"{'model__C': 0.1, 'model__gamma': 'scale', 'mo...",0.920000,0.045216
...,...,...,...,...
1985,Random Forest,"{'model__criterion': 'log_loss', 'model__max_d...",0.946667,0.040000
1986,Random Forest,"{'model__criterion': 'log_loss', 'model__max_d...",0.960000,0.024944
1987,Random Forest,"{'model__criterion': 'log_loss', 'model__max_d...",0.966667,0.021082
1988,Random Forest,"{'model__criterion': 'log_loss', 'model__max_d...",0.960000,0.024944


In [7]:
randomized_scores_df

,Model,Parameters,Mean CV Score,Std CV Score
0,Logistic Regression,{'model__max_iter': 50},0.926667,0.057349
1,Logistic Regression,{'model__max_iter': 100},0.926667,0.057349
2,Logistic Regression,{'model__max_iter': 200},0.926667,0.057349
3,SVM,"{'model__max_iter': 50, 'model__kernel': 'rbf'...",0.966667,0.029814
4,SVM,"{'model__max_iter': 100, 'model__kernel': 'sig...",0.886667,0.080554
5,SVM,"{'model__max_iter': 50, 'model__kernel': 'rbf'...",0.953333,0.033993
6,SVM,"{'model__max_iter': 100, 'model__kernel': 'lin...",0.966667,0.021082
7,SVM,"{'model__max_iter': 100, 'model__kernel': 'sig...",0.200000,0.069921
8,SVM,"{'model__max_iter': 50, 'model__kernel': 'sigm...",0.886667,0.080554
9,SVM,"{'model__max_iter': 50, 'model__kernel': 'sigm...",0.960000,0.038873


In [ ]:
grid_scores_df.sort_values(by = ["Mean CV Score", "Std CV Score"], ascending = [False, True]).head(25)
# Best Candidate Model to start from is... SVC(C = 10, gamma = 'scale', kernal = 'rbf', max_iter = 100)

,Model,Parameters,Mean CV Score,Std CV Score
57,SVM,"{'model__C': 10, 'model__gamma': 'scale', 'mod...",0.973333,0.032660
58,SVM,"{'model__C': 10, 'model__gamma': 'scale', 'mod...",0.973333,0.032660
59,SVM,"{'model__C': 10, 'model__gamma': 'scale', 'mod...",0.973333,0.032660
97,SVM,"{'model__C': 100, 'model__gamma': 'auto', 'mod...",0.973333,0.038873
98,SVM,"{'model__C': 100, 'model__gamma': 'auto', 'mod...",0.973333,0.038873
102,Random Forest,"{'model__criterion': 'gini', 'model__max_depth...",0.966667,0.021082
103,Random Forest,"{'model__criterion': 'gini', 'model__max_depth...",0.966667,0.021082
107,Random Forest,"{'model__criterion': 'gini', 'model__max_depth...",0.966667,0.021082
108,Random Forest,"{'model__criterion': 'gini', 'model__max_depth...",0.966667,0.021082
112,Random Forest,"{'model__criterion': 'gini', 'model__max_depth...",0.966667,0.021082


In [ ]:
randomized_scores_df.sort_values(by = ["Mean CV Score", "Std CV Score"], ascending = [False, True]).head(25)
# Best Candidate Model (in this random set) to start from is... RandomForestClassifier(
#     n_estimators = 10,
#     min_samples_split = 2,
#     min_samples_leaf = 3,
#     max_features = 'log2',
#     max_depth = 4,
#     criterion = 'entropy'
# )


,Model,Parameters,Mean CV Score,Std CV Score
19,Random Forest,"{'model__n_estimators': 50, 'model__min_sample...",0.966667,0.021082
23,Random Forest,"{'model__n_estimators': 10, 'model__min_sample...",0.966667,0.021082
26,Random Forest,"{'model__n_estimators': 50, 'model__min_sample...",0.966667,0.021082
28,Random Forest,"{'model__n_estimators': 50, 'model__min_sample...",0.966667,0.021082
3,SVM,"{'model__max_iter': 50, 'model__kernel': 'rbf'...",0.966667,0.029814
13,SVM,"{'model__max_iter': 100, 'model__kernel': 'pol...",0.966667,0.029814
6,SVM,"{'model__max_iter': 100, 'model__kernel': 'lin...",0.966667,0.021082
20,Random Forest,"{'model__n_estimators': 100, 'model__min_sampl...",0.960000,0.024944
21,Random Forest,"{'model__n_estimators': 10, 'model__min_sample...",0.960000,0.024944
31,Random Forest,"{'model__n_estimators': 10, 'model__min_sample...",0.960000,0.024944
